# 09_04_オンライン_softmax_の内部状態

コード：`ch09/blocks/09_04_オンライン_softmax_の内部状態.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# オンライン softmax：16 個のスコアを Bc 個ずつ読み、m・ℓ・o を更新する
import numpy as np

rng = np.random.default_rng(0)
scores = np.round(rng.normal(0, 2, 16), 2)
values = np.round(rng.normal(0, 1, 16), 2)           # 値ベクトルは 1 次元にして読みやすく

w = np.exp(scores - scores.max()); w /= w.sum()
truth = w @ values                                    # 素朴に計算した正解

def online(sc, vs, Bc):
    m, l, o, log = -np.inf, 0.0, 0.0, []
    for start in range(0, len(sc), Bc):
        x, v = sc[start:start + Bc], vs[start:start + Bc]
        m_old, l_old = m, l
        m = max(m, x.max())
        l = l_old * np.exp(m_old - m) + np.exp(x - m).sum()
        rescale = l_old * np.exp(m_old - m) / l       # 古い基準 → 新しい基準
        o = o * rescale + (np.exp(x - m) / l) @ v
        log.append((start + len(x), m, np.exp(m_old - m), l, rescale, o))
    return o, log

orders = {"最大値が後ろに来る": np.argsort(scores),        # 昇順に並べ替え
          "最大値が前に来る": np.argsort(-scores)}         # 降順に並べ替え
for name, idx in orders.items():
    o, log = online(scores[idx], values[idx], Bc=2)
    print(f"{name}（Bc=2）  最終の |o − 真値| = {abs(o - truth):.1e}")
    for done, m, shift, l, r, o_t in log:
        print(f"  {done:2d}/16  m={m:5.2f}  e^(m_old−m)={shift:.3f}  ℓ={l:7.3f}  "
              f"rescale={r:.3f}  |o−真値|={abs(o_t - truth):.3f}")

for Bc in (1, 4, 16):                                  # ブロック幅を変えても答えは同じ
    o, _ = online(scores, values, Bc)
    print(f"Bc={Bc:2d}  |o − 真値| = {abs(o - truth):.1e}")
